# NB16 · Numbers, Bits, Bytes & Unicode

`intermediate/` · notebook 16 of 32 · 34 questions + a mini-project

This notebook looks underneath the values you've used since NB01. How does
Python store a number, and when can it not store it exactly? What is text,
once it's saved to a file or sent over a network?

You'll meet four ideas. **Exact numbers**: `int` never overflows, `float`
is a binary approximation, and `Decimal` and `Fraction` are exact when you
need them (money, recipes). **Bits**: number bases and the bitwise
operators used for flags and masks. **Bytes**: the difference between
`str` (characters) and `bytes` (raw numbers 0 to 255), and how `encode` and
`decode` cross between them. **Unicode**: why `"é"` can be stored two
different ways, and how to compare text safely.

The modules here (`decimal`, `fractions`, `unicodedata`, `struct`,
`ipaddress`) all come with Python.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB15: standard-library basics

## What you'll practise

- Integers of any size; floats as IEEE 754; math.isclose; inf and nan
- Decimal for money: quantize, ROUND_HALF_UP; Fraction
- Number bases: bin, oct, hex, int(s, base)
- Bitwise operators & | ^ ~ << >>; flags and masks; int.bit_count()
- str vs bytes; encode and decode; UTF-8 byte lengths
- unicodedata, NFC/NFD normalisation, casefold for comparing text
- bytearray; struct for packing binary records

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB16-Q01 · L1 · Decimal · Predict

Floats against `Decimal`. Predict the output.

**What does this print?**

```python
from decimal import Decimal

print(0.1 + 0.2)
print(Decimal("0.1") + Decimal("0.2"))
print(0.1 + 0.2 == 0.3, Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))
```

<details><summary><b>Hint</b></summary>

A float stores the nearest *binary* fraction, and 0.1 has no exact binary form. A `Decimal` built from text stores the decimal digits exactly.

</details>

<details><summary><b>Expected output</b></summary>

```text
0.30000000000000004
0.3
False True
```

</details>

<details><summary><b>Solution</b></summary>

`0.1` and `0.2` are each stored slightly off, and the errors show up as `0.30000000000000004`. `Decimal("0.1")` stores exactly one tenth, so the sum is exactly `0.3` and the comparison is `True`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Seniors use `float` for measurements (where tiny errors don't matter) and `Decimal` for money (where they do). The cost: `Decimal` is slower and must be built from **text**, as Q17 shows.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB16-Q02 · L1 · Big integers · Write

A chessboard has 64 squares. If each square holds double the grains of rice
of the one before, the total is `2 ** 64 - 1`. Print the total, then how many
digits it has.

**Starting code (already in the cell below):**

```python
grains = 2 ** 64 - 1
```

<details><summary><b>Hint</b></summary>

Python ints have no size limit. To count digits, turn the number into text.

</details>

<details><summary><b>Expected output</b></summary>

```text
18446744073709551615
20
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(grains)
print(len(str(grains)))
```

Python's `int` grows as large as it needs to, so there's no overflow and no rounding: every digit is exact. `str(grains)` gives the digits as text, and `len` counts them.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
print(grains)
print(len(str(grains)))
```

Same code. This number is the largest *unsigned 64-bit* integer, where C or Java would wrap around to 0. In Python it's an ordinary int. `grains.bit_length()` tells you how many bits it needs: 64.

</details>

In [ ]:
grains = 2 ** 64 - 1

# your code here


### NB16-Q03 · L1 · Number bases · Write

Print `n` in binary, octal and hexadecimal on one line. Then convert the
binary text `"101010"` and the hex text `"2a"` back to ints and print them.

```text
0b101010 0o52 0x2a
42 42
```

**Starting code (already in the cell below):**

```python
n = 42
```

<details><summary><b>Hint</b></summary>

`bin`, `oct` and `hex` turn an int into text. `int(text, base)` goes the other way.

</details>

<details><summary><b>Expected output</b></summary>

```text
0b101010 0o52 0x2a
42 42
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(bin(n), oct(n), hex(n))
print(int("101010", 2), int("2a", 16))
```

The same number can be written in any base: 42 is `101010` in base 2, `52` in base 8 and `2a` in base 16. The prefixes `0b`, `0o` and `0x` say which base is meant. `int(text, base)` reads text written in that base.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
print(f"{n:#b} {n:#o} {n:#x}")
print(int("101010", 2), int("2a", 16))
```

f-string format codes `b`, `o` and `x` do the same job, and `#` adds the prefix. Without `#` you get bare digits, which is what you usually want for padded output like `f"{n:08b}"` (Q07).

</details>

In [ ]:
n = 42

# your code here


### NB16-Q04 · L1 · str and bytes · Write

Encode `message` to UTF-8 bytes and print the bytes and their type. Then
decode the bytes back to text and print that.

```text
b'Hi!'
<class 'bytes'>
Hi!
```

**Starting code (already in the cell below):**

```python
message = "Hi!"
```

<details><summary><b>Hint</b></summary>

Text has an `.encode("utf-8")` method. Bytes have a `.decode("utf-8")` method.

</details>

<details><summary><b>Expected output</b></summary>

```text
b'Hi!'
<class 'bytes'>
Hi!
```

</details>

<details><summary><b>Solution</b></summary>

```python
data = message.encode("utf-8")
print(data)
print(type(data))
print(data.decode("utf-8"))
```

A `str` holds characters; `bytes` hold raw numbers from 0 to 255, which is what files and networks actually store. `encode` turns text into bytes using a named encoding, and `decode` turns bytes back into text. Python shows printable bytes as characters after a `b` prefix.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
data = message.encode()
print(data)
print(type(data))
print(data.decode())
```

UTF-8 is the default for `encode()` and `decode()`, so seniors often leave it out here. They do still write `encoding="utf-8"` in `open()` (NB13), because that default depends on the operating system.

</details>

In [ ]:
message = "Hi!"

# your code here


### NB16-Q05 · L1 · Fraction · Write

A recipe uses 1/3 cup of milk and 1/6 cup of cream. Print the total as an
exact fraction, then as a float.

```text
1/2
0.5
```

**Starting code (already in the cell below):**

```python
from fractions import Fraction
```

<details><summary><b>Hint</b></summary>

`Fraction(1, 3)` is exactly one third. Fractions add like numbers, and `float()` converts one.

</details>

<details><summary><b>Expected output</b></summary>

```text
1/2
0.5
```

</details>

<details><summary><b>Solution</b></summary>

```python
total = Fraction(1, 3) + Fraction(1, 6)
print(total)
print(float(total))
```

A `Fraction` stores a whole-number numerator and denominator, so 1/3 is exact (as a float it would be `0.333...`, cut off). Fractions reduce themselves: 1/3 + 1/6 is 3/6, shown as `1/2`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
MILK = Fraction(1, 3)
CREAM = Fraction(1, 6)
total = MILK + CREAM
print(total)
print(float(total))
```

Named amounts read like the recipe. `Fraction` is the right tool for exact ratios (scaling recipes, music timings, probabilities); for money, `Decimal` matches how prices are written.

</details>

In [ ]:
from fractions import Fraction

# your code here


---

## L2 · Easy

### NB16-Q06 · L2 · Decimal · Write

Print `round(2.675, 2)` using floats. Then round `price` to 2 decimal places
with `Decimal`, rounding halves **up**, and print that.

```text
2.67
2.68
```

**Starting code (already in the cell below):**

```python
from decimal import Decimal, ROUND_HALF_UP

price = Decimal("2.675")
```

<details><summary><b>Hint</b></summary>

`price.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)` rounds to cents.

</details>

<details><summary><b>Expected output</b></summary>

```text
2.67
2.68
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(round(2.675, 2))
print(price.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))
```

The float `2.675` is really stored as `2.67499999...`, so `round` goes down. The `Decimal` is exactly `2.675`, and `ROUND_HALF_UP` sends an exact half up to `2.68`, the way shops and accountants expect. `quantize` means "round to the same number of places as this example".

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
CENT = Decimal("0.01")
print(round(2.675, 2))
print(price.quantize(CENT, rounding=ROUND_HALF_UP))
```

A `CENT` constant names the precision once. `Decimal`'s own default is `ROUND_HALF_EVEN` (banker's rounding), so seniors always pass the rounding mode explicitly in money code: the rule belongs in the code, not in a default.

</details>

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

price = Decimal("2.675")

# your code here


### NB16-Q07 · L2 · Number bases · Write

Print `n` as exactly 8 binary digits (with leading zeros), then convert
that text back to an int and print it.

```text
11001010
202
```

**Starting code (already in the cell below):**

```python
n = 202
```

<details><summary><b>Hint</b></summary>

The format code `08b` means binary, padded with zeros to 8 characters. `int(text, 2)` reads binary text.

</details>

<details><summary><b>Expected output</b></summary>

```text
11001010
202
```

</details>

<details><summary><b>Solution</b></summary>

```python
bits = format(n, "08b")
print(bits)
print(int(bits, 2))
```

`format(n, "08b")` writes `n` in base 2, padded with zeros to 8 characters. That's one byte, with the most significant bit on the left. `int(bits, 2)` reads it back.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
bits = f"{n:08b}"
print(bits)
print(int(bits, 2))
```

The f-string version uses the same `08b` code. Fixed-width binary is how seniors print flags and bytes, so every bit lines up with the one above it.

</details>

In [ ]:
n = 202

# your code here


### NB16-Q08 · L2 · inf and nan · Predict

Special float values. Predict the output.

**What does this print?**

```python
import math

nan = float("nan")
inf = float("inf")
print(nan == nan, nan != nan, math.isnan(nan))
print(inf > 10 ** 300, inf - inf, -inf < -10 ** 300)
```

<details><summary><b>Hint</b></summary>

`nan` means "not a number" and is never equal to anything, *including itself*. Infinity minus infinity has no sensible answer.

</details>

<details><summary><b>Expected output</b></summary>

```text
False True True
True nan True
```

</details>

<details><summary><b>Solution</b></summary>

`nan == nan` is `False` and `nan != nan` is `True`: that's the IEEE 754 rule for "not a number", so `math.isnan` is the only reliable check. `inf` is bigger than any number and `-inf` smaller than any. `inf - inf` is undefined, so it gives `nan`.

</details>

<details><summary><b>Senior dev solution</b></summary>

A single `nan` in a list quietly breaks sorting, `max()` and `==` checks. Seniors filter with `math.isnan` (or `math.isfinite` to also drop infinities) as data arrives.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB16-Q09 · L2 · Decimal · Error

Which error does mixing these two number types raise?

**Which error does this raise?**

```python
from decimal import Decimal

price = Decimal("1.10")
print(price + 0.2)
```

<details><summary><b>Hint</b></summary>

Should Python silently mix an exact number with an inexact one?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: `Decimal` refuses to do arithmetic with a `float`. The float `0.2` is already slightly wrong, and quietly mixing it in would make the exact `Decimal` inexact too. (`Decimal` with `int` is fine: ints are exact.)

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from decimal import Decimal

price = Decimal("1.10")
print(price + Decimal("0.2"))
```

The error is a feature: it stops floats leaking into money code. Seniors make every amount a `Decimal` the moment it enters the program, from text.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB16-Q10 · L2 · Floats · Write

Print whether `total` is close to `0.6`, then whether `balance` is close to
zero. Both answers should be `True`.

```text
True True
```

**Starting code (already in the cell below):**

```python
import math

total = 0.1 + 0.2 + 0.3
balance = 0.1 + 0.2 - 0.3
```

<details><summary><b>Hint</b></summary>

`math.isclose` compares *relative* to the size of the numbers, and nothing is relatively close to zero. It takes an `abs_tol=` argument for that case.

</details>

<details><summary><b>Expected output</b></summary>

```text
True True
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(math.isclose(total, 0.6), math.isclose(balance, 0, abs_tol=1e-9))
```

`total` is `0.6000000000000001` and `balance` is about `5.5e-17`, not 0. `isclose` by default allows a tiny *relative* difference, which works for 0.6. But a relative difference from 0 is always 100%, so comparing with zero needs an absolute tolerance (`abs_tol`).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
TOLERANCE = 1e-9
print(math.isclose(total, 0.6), math.isclose(balance, 0, abs_tol=TOLERANCE))
```

Pick `abs_tol` from the problem: a balance in pounds might use `0.005`, a physics result `1e-12`. Naming the tolerance documents that choice.

</details>

In [ ]:
import math

total = 0.1 + 0.2 + 0.3
balance = 0.1 + 0.2 - 0.3

# your code here


### NB16-Q11 · L2 · Bitwise operators · Predict

Bitwise operators on small numbers. Predict the output.

**What does this print?**

```python
print(12 & 10, 12 | 10, 12 ^ 10)
print(1 << 4, 200 >> 3, ~5)
```

<details><summary><b>Hint</b></summary>

Write 12 as `1100` and 10 as `1010`, then compare bit by bit. `<<` and `>>` shift bits left and right. `~x` is `-x - 1`.

</details>

<details><summary><b>Expected output</b></summary>

```text
8 14 6
16 25 -6
```

</details>

<details><summary><b>Solution</b></summary>

`1100 & 1010` keeps bits set in both: `1000` = 8. `|` keeps bits set in either: `1110` = 14. `^` keeps bits set in exactly one: `0110` = 6. `1 << 4` is 1 times 2⁴ = 16, and `200 >> 3` is `200 // 8` = 25. `~5` flips every bit, which for Python's ints means `-6`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Shifts are multiplying and dividing by powers of 2, but seniors write `* 16` when they mean arithmetic and `<< 4` only when they mean bit positions. Code should say what it means.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB16-Q12 · L2 · Bit flags · Write

`open_days` stores the days a shop opens as bits: bit 0 is Monday up to
bit 6 for Sunday. Print how many days it opens, then whether it opens on
Saturday (bit 5) and on Sunday (bit 6).

```text
6
Saturday: True
Sunday: False
```

**Starting code (already in the cell below):**

```python
open_days = 0b0111111
```

<details><summary><b>Hint</b></summary>

`int.bit_count()` counts the 1 bits. `1 << 5` is a number with only bit 5 set; `&` it with `open_days` and check whether the result is non-zero.

</details>

<details><summary><b>Expected output</b></summary>

```text
6
Saturday: True
Sunday: False
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(open_days.bit_count())
print("Saturday:", (open_days & (1 << 5)) != 0)
print("Sunday:", (open_days & (1 << 6)) != 0)
```

`0b0111111` has bits 0 to 5 set, so `bit_count()` is 6. `1 << 5` is a *mask* with only Saturday's bit set; `&` keeps that one bit if `open_days` has it and gives 0 otherwise.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
SATURDAY = 1 << 5
SUNDAY = 1 << 6
print(open_days.bit_count())
print("Saturday:", bool(open_days & SATURDAY))
print("Sunday:", bool(open_days & SUNDAY))
```

Named masks make the intent obvious. `bool(x & MASK)` reads as "is this bit set?". For real code with many flags, `enum.Flag` (NB21) gives the same bits with names and a readable `repr`.

</details>

In [ ]:
open_days = 0b0111111

# your code here


### NB16-Q13 · L2 · Number bases · Fix

`channel` is the red part of the hex colour `#ff8800`. This should print
`255`, but it crashes. Fix it.

**Buggy code (copied into the cell below):**

```python
channel = "ff"
print(int(channel))
```

<details><summary><b>Hint</b></summary>

By default `int()` reads base 10 text. Which base is `ff` written in?

</details>

<details><summary><b>Expected output</b></summary>

```text
255
```

</details>

<details><summary><b>Solution</b></summary>

```python
channel = "ff"
print(int(channel, 16))
```

`int("ff")` raises `ValueError`, because `f` isn't a base-10 digit. Passing `16` as the base reads it as hexadecimal: `f` is 15, so `ff` is 15 × 16 + 15 = 255.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
HEX = 16
channel = "ff"
print(int(channel, HEX))
```

A tiny constant can help readers who don't know their bases, but `16` is widely understood. The real senior habit is reaching for the `base` argument instead of hand-written digit maths.

</details>

In [ ]:
channel = "ff"
print(int(channel))


---

## L3 · Intermediate

### NB16-Q14 · L3 · Bit flags · Write

`perms` holds file permission flags. Print whether it allows reading and
writing. Then grant `WRITE`, revoke `EXEC`, and print the final value as a
number and as 3 binary digits.

```text
read: True write: False
3 011
```

**Starting code (already in the cell below):**

```python
READ = 1
WRITE = 2
EXEC = 4

perms = READ | EXEC
```

<details><summary><b>Hint</b></summary>

Test a flag with `&`. Grant one with `|=`. Revoke one with `&= ~FLAG`, which keeps every bit except that one.

</details>

<details><summary><b>Expected output</b></summary>

```text
read: True write: False
3 011
```

</details>

<details><summary><b>Solution</b></summary>

```python
print("read:", (perms & READ) != 0, "write:", (perms & WRITE) != 0)
perms |= WRITE
perms &= ~EXEC
print(perms, format(perms, "03b"))
```

Each flag is a different bit (1, 2, 4), so `|` combines them without clashing: `READ | EXEC` is `101`. `|= WRITE` sets the middle bit, giving `111`. `~EXEC` is every bit *except* EXEC's, so `&= ~EXEC` clears only that bit, leaving `011` = 3.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
print("read:", bool(perms & READ), "write:", bool(perms & WRITE))
perms |= WRITE
perms &= ~EXEC
print(perms, f"{perms:03b}")
```

This is how Unix file modes work (`chmod 755` is three sets of these bits). Seniors use `^=` to *toggle* a flag, and never use `-` to remove one: `perms - EXEC` breaks if the flag wasn't set.

</details>

In [ ]:
READ = 1
WRITE = 2
EXEC = 4

perms = READ | EXEC

# your code here


### NB16-Q15 · L3 · UTF-8 · Predict

Characters versus bytes. Predict the output.

**What does this print?**

```python
text = "café"
print(len(text), len(text.encode("utf-8")))
word = "日本"
print(len(word), len(word.encode("utf-8")))
```

<details><summary><b>Hint</b></summary>

`len` of a `str` counts characters. UTF-8 uses 1 byte for plain English letters, 2 for accented Latin letters and 3 for most Chinese and Japanese characters.

</details>

<details><summary><b>Expected output</b></summary>

```text
4 5
2 6
```

</details>

<details><summary><b>Solution</b></summary>

`café` is 4 characters, but `é` needs 2 bytes in UTF-8, so it's 5 bytes. `日本` is 2 characters of 3 bytes each: 6 bytes. Character count and byte count only match for plain ASCII text.

</details>

<details><summary><b>Senior dev solution</b></summary>

This matters whenever a limit is in **bytes**: database columns, network packets, SMS messages. Seniors check `len(text.encode())` against those limits, never `len(text)`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB16-Q16 · L3 · str and bytes · Error

These bytes came from an old file. Which error does decoding them raise?

**Which error does this raise?**

```python
data = b"caf\xe9"
print(data.decode("utf-8"))
```

<details><summary><b>Hint</b></summary>

In UTF-8, `é` is two bytes. Here it's one byte, `e9`. Is that valid UTF-8?

</details>

<details><summary><b>Expected output</b></summary>

```text
UnicodeDecodeError
```

</details>

<details><summary><b>Solution</b></summary>

`UnicodeDecodeError`: the byte `e9` is how the old Latin-1 encoding stores `é`, but in UTF-8 a byte like `e9` must be followed by more bytes. Decoding with the wrong encoding either fails like this or, worse, gives garbled text.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
data = b"caf\xe9"
print(data.decode("latin-1"))
```

Bytes don't record their encoding; you have to know it. Seniors find out from the source (a header, the docs) and decode with that. `errors="replace"` is a last resort that shows `�` instead of crashing.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB16-Q17 · L3 · Decimal · Fix

Three items at `19.99` each should cost `59.97`, but this prints a long,
wrong number. Fix it.

**Buggy code (copied into the cell below):**

```python
from decimal import Decimal

price = Decimal(19.99)
print(price * 3)
```

<details><summary><b>Hint</b></summary>

What exactly is the value being handed to `Decimal` here: the text `19.99`, or a float?

</details>

<details><summary><b>Expected output</b></summary>

```text
59.97
```

</details>

<details><summary><b>Solution</b></summary>

```python
from decimal import Decimal

price = Decimal("19.99")
print(price * 3)
```

`Decimal(19.99)` faithfully copies the *float* `19.99`, which is really `19.989999999999998436805981327779591083526611328125`. The float's error is now exact and permanent. `Decimal("19.99")` builds the value from text, so it's exactly 19.99.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from decimal import Decimal

PRICE = Decimal("19.99")
quantity = 3
print(PRICE * quantity)
```

The rule: build a `Decimal` from a **string** (or an int), never from a float. Prices usually arrive as text anyway (forms, CSV, JSON strings), so pass the text straight in.

</details>

In [ ]:
from decimal import Decimal

price = Decimal(19.99)
print(price * 3)


### NB16-Q18 · L3 · Number bases · Write

Split the hex colour into its red, green and blue values (0 to 255), print
them, then rebuild the colour in lowercase hex.

```text
30 144 255
#1e90ff
```

**Starting code (already in the cell below):**

```python
colour = "#1E90FF"
```

<details><summary><b>Hint</b></summary>

Each pair of hex digits is one channel: slice them out and use `int(..., 16)`. The format code `02x` writes two lowercase hex digits.

</details>

<details><summary><b>Expected output</b></summary>

```text
30 144 255
#1e90ff
```

</details>

<details><summary><b>Solution</b></summary>

```python
red = int(colour[1:3], 16)
green = int(colour[3:5], 16)
blue = int(colour[5:7], 16)
print(red, green, blue)
print(f"#{red:02x}{green:02x}{blue:02x}")
```

Two hex digits hold one byte, 0 to 255. `1E` is 30, `90` is 144 and `FF` is 255. On the way back, `02x` pads each value to two digits, so a channel of 5 becomes `05` and the colour keeps its six digits.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
value = int(colour[1:], 16)
red, green, blue = value >> 16, (value >> 8) & 0xFF, value & 0xFF
print(red, green, blue)
print(f"#{value:06x}")
```

Seniors read the whole colour as one 24-bit number, then shift and mask out each byte. `0xFF` is a mask of 8 one-bits. This is exactly how graphics code packs pixels.

</details>

In [ ]:
colour = "#1E90FF"

# your code here


### NB16-Q19 · L3 · struct · Write

Pack a sensor reading into bytes with `struct`, using the format `"<HhI"`
(little-endian: unsigned 2-byte id, signed 2-byte temperature in tenths of a
degree, unsigned 4-byte timestamp). Print the number of bytes, the bytes as
hex pairs, then unpack them and print the tuple.

```text
8
07 00 d6 ff 08 a7 55 69
(7, -42, 1767221000)
```

**Starting code (already in the cell below):**

```python
import struct

sensor_id = 7
temp_tenths = -42
timestamp = 1767221000
```

<details><summary><b>Hint</b></summary>

`struct.pack(format, *values)` returns bytes; `struct.unpack(format, data)` returns a tuple. Bytes have a `.hex(" ")` method.

</details>

<details><summary><b>Expected output</b></summary>

```text
8
07 00 d6 ff 08 a7 55 69
(7, -42, 1767221000)
```

</details>

<details><summary><b>Solution</b></summary>

```python
packed = struct.pack("<HhI", sensor_id, temp_tenths, timestamp)
print(len(packed))
print(packed.hex(" "))
print(struct.unpack("<HhI", packed))
```

`H` is 2 bytes unsigned, `h` 2 bytes signed, `I` 4 bytes unsigned: 8 bytes in total. `<` means little-endian, lowest byte first, so id 7 is `07 00`. -42 is stored in two's complement as `d6 ff`. Unpacking with the same format gives the original values back.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
READING = struct.Struct("<HhI")
packed = READING.pack(sensor_id, temp_tenths, timestamp)
print(READING.size)
print(packed.hex(" "))
print(READING.unpack(packed))
```

A `struct.Struct` object compiles the format once and names it, so the writer and reader can't drift apart. Always state the byte order (`<` or `>`): without it, `struct` uses the machine's native order and padding, which differs between computers.

</details>

In [ ]:
import struct

sensor_id = 7
temp_tenths = -42
timestamp = 1767221000

# your code here


### NB16-Q20 · L3 · str and bytes · Error

A program tries to patch one byte of a message. Which error does this raise?

**Which error does this raise?**

```python
data = b"PING"
data[1] = 79
print(data)
```

<details><summary><b>Hint</b></summary>

Is `bytes` changeable like a list, or fixed like a `str`?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: `bytes` is immutable, exactly like `str`. You can *read* `data[1]` (it gives the int `73`, the code for `I`), but you can't assign to it. To change bytes in place you need a `bytearray` (next question).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
data = bytearray(b"PING")
data[1] = 79
print(bytes(data))
```

Seniors keep finished messages as immutable `bytes`, so nothing can change them by accident, and use a `bytearray` only while building one. `bytes(data)` freezes it again when it's done.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB16-Q21 · L3 · bytearray · Write

Turn `packet` from `PING` into `PONG` by changing one byte, then append a
checksum byte (the sum of all the bytes, modulo 256). Print the packet and
its bytes as a list of ints.

```text
bytearray(b'PONG4')
[80, 79, 78, 71, 52]
```

**Starting code (already in the cell below):**

```python
packet = bytearray(b"PING")
```

<details><summary><b>Hint</b></summary>

A `bytearray` is a mutable `bytes`: assign an int to an index, and `append` an int. `ord("O")` gives the byte value for `O`.

</details>

<details><summary><b>Expected output</b></summary>

```text
bytearray(b'PONG4')
[80, 79, 78, 71, 52]
```

</details>

<details><summary><b>Solution</b></summary>

```python
packet[1] = ord("O")
packet.append(sum(packet) % 256)
print(packet)
print(list(packet))
```

`bytes` can't change, but a `bytearray` can. Each item is an int from 0 to 255, so you assign `ord("O")` (79), not the text `"O"`. The bytes add up to 308, and `308 % 256` is 52, which happens to be the character `4`, so Python shows it as `4`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
packet[1:2] = b"O"
checksum = sum(packet) % 256
packet.append(checksum)
print(packet)
print(list(packet))
```

Slice assignment with a bytes literal (`b"O"`) avoids the `ord` call. Seniors build binary messages in a `bytearray` and convert to `bytes` once at the end, which avoids copying on every change.

</details>

In [ ]:
packet = bytearray(b"PING")

# your code here


### NB16-Q22 · L3 · Floats · Predict

Where floats run out. Predict the output.

**What does this print?**

```python
print(2 ** 53 + 1 == 2 ** 53, float(2 ** 53 + 1) == float(2 ** 53))
print(1e308 * 10, 10 ** 400 > 1e308)
```

<details><summary><b>Hint</b></summary>

A float has 53 bits for its digits. Above 2⁵³, it can't store every whole number. The largest float is about `1.8e308`.

</details>

<details><summary><b>Expected output</b></summary>

```text
False True
inf True
```

</details>

<details><summary><b>Solution</b></summary>

As ints, `2**53 + 1` and `2**53` differ, so the first is `False`. As floats, `2**53 + 1` can't be stored and rounds to `2**53`, so they're equal: `True`. `1e308 * 10` is past the largest float and becomes `inf`. `10 ** 400` is an int, and Python compares it exactly: `True`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is why IDs and counters must stay `int`, and why JSON numbers from JavaScript (which uses floats) lose precision above 2⁵³. Seniors send big IDs as strings for exactly this reason.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB16-Q23 · L3 · Fraction · Write

Turn each decimal measurement (in inches) into an exact fraction and print
them on one line. Then print the total, and the total as a mixed number.

```text
1/8 3/8 5/4
total: 7/4
mixed: 1 3/4
```

**Starting code (already in the cell below):**

```python
from fractions import Fraction

measurements = ["0.125", "0.375", "1.25"]
```

<details><summary><b>Hint</b></summary>

`Fraction("0.125")` reads decimal text exactly. For the mixed number, `divmod(numerator, denominator)` gives the whole part and what's left.

</details>

<details><summary><b>Expected output</b></summary>

```text
1/8 3/8 5/4
total: 7/4
mixed: 1 3/4
```

</details>

<details><summary><b>Solution</b></summary>

```python
parts = [Fraction(text) for text in measurements]
print(" ".join(str(part) for part in parts))
total = sum(parts)
print("total:", total)
whole, rest = divmod(total.numerator, total.denominator)
print("mixed:", whole, Fraction(rest, total.denominator))
```

A `Fraction` built from text is exact, the same as a `Decimal`. `sum` works because fractions add to fractions: 1/8 + 3/8 + 5/4 = 7/4. `divmod(7, 4)` is `(1, 3)`: one whole inch and 3/4 left over.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
parts = [Fraction(text) for text in measurements]
total = sum(parts, start=Fraction(0))
whole, rest = divmod(total, 1)
print(*parts)
print("total:", total)
print("mixed:", whole, rest)
```

`divmod` works on fractions directly: `divmod(7/4, 1)` gives the whole part and the fractional rest. Starting `sum` at `Fraction(0)` keeps the result a `Fraction` even for an empty list.

</details>

In [ ]:
from fractions import Fraction

measurements = ["0.125", "0.375", "1.25"]

# your code here


### NB16-Q24 · L3 · Number bases · Refactor

This builds the binary digits of `n` by hand. Refactor it with built-in
tools, keeping the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
n = 37
digits = ""
while n > 0:
    digits = str(n % 2) + digits
    n = n // 2
print(digits)
print(len(digits))
```

<details><summary><b>Hint</b></summary>

`bin(n)` gives the digits with a `0b` prefix. Slicing can remove it.

</details>

<details><summary><b>Expected output</b></summary>

```text
100101
6
```

</details>

<details><summary><b>Solution</b></summary>

```python
n = 37
bits = bin(n)[2:]
print(bits)
print(len(bits))
```

The loop is the textbook algorithm (repeatedly divide by 2 and collect remainders), and `bin` does exactly that in C. `[2:]` drops the `0b`. As a bonus, the original destroyed `n`; this version doesn't.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
n = 37
print(f"{n:b}")
print(n.bit_length())
```

`f"{n:b}"` gives the digits with no prefix to strip, and `n.bit_length()` answers "how many bits?" without building any text. Watch one edge: for 0 the hand loop printed nothing, `bin` gives `0`.

</details>

In [ ]:
n = 37
digits = ""
while n > 0:
    digits = str(n % 2) + digits
    n = n // 2
print(digits)
print(len(digits))


---

## L4 · Somewhat Difficult

### NB16-Q25 · L4 · Unicode normalisation · Write

`typed` came from a keyboard and `stored` from a database. They're the same
word, but stored differently. Print whether they're equal as they are, then
whether they're equal after normalising both to NFC and ignoring case.

```text
False
True
```

**Starting code (already in the cell below):**

```python
import unicodedata

typed = "Caf\u00e9"      # é as one character
stored = "CAFE\u0301"    # E followed by a combining accent
```

<details><summary><b>Hint</b></summary>

`unicodedata.normalize("NFC", text)` joins letter + accent into single characters. `.casefold()` is the strongest way to ignore case.

</details>

<details><summary><b>Expected output</b></summary>

```text
False
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(typed == stored)
clean_typed = unicodedata.normalize("NFC", typed).casefold()
clean_stored = unicodedata.normalize("NFC", stored).casefold()
print(clean_typed == clean_stored)
```

`stored` ends with a plain `E` followed by U+0301, a *combining* accent that draws on top of the letter before it. It looks identical but uses different code points. NFC normalisation combines `E` + accent into the single character `É`, and `casefold` then turns both words into `café`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def same_text(a, b):
    """Compare text the way a human reader would."""
    def key(s):
        return unicodedata.normalize("NFC", s).casefold()
    return key(a) == key(b)

print(typed == stored)
print(same_text(typed, stored))
```

Seniors normalise text once, at the point it enters the system (form, file, API), and store it in NFC. Then a plain `==` works everywhere else. A `same_text` helper documents the rule for the few places that compare raw input.

</details>

In [ ]:
import unicodedata

typed = "Caf\u00e9"      # é as one character
stored = "CAFE\u0301"    # E followed by a combining accent

# your code here


### NB16-Q26 · L4 · Unicode normalisation · Predict

One letter, two spellings. Predict the output.

**What does this print?**

```python
import unicodedata

a = "\u00e9"
b = "e\u0301"
print(a, b, a == b)
print(len(a), len(b))
print(unicodedata.normalize("NFC", b) == a, len(unicodedata.normalize("NFD", a)))
```

<details><summary><b>Hint</b></summary>

`\u00e9` is the single character `é`. `e\u0301` is a plain `e` plus a combining accent. They look the same when printed.

</details>

<details><summary><b>Expected output</b></summary>

```text
é é False
1 2
True 2
```

</details>

<details><summary><b>Solution</b></summary>

Both print as `é`, but `a` is one code point and `b` is two, so `==` is `False` and the lengths are 1 and 2. NFC (*composed*) turns `b` into the single-character form, so it equals `a`. NFD (*decomposed*) splits `a` into letter + accent: length 2.

</details>

<details><summary><b>Senior dev solution</b></summary>

macOS file names, copy-pasted text and some keyboards produce NFD; most of the web uses NFC. That mismatch is behind "the file is right there but Python can't find it" bugs. Normalise before you compare or search.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB16-Q27 · L4 · unicodedata · Write

Remove the accents from `dish` (for a search index), keeping the letters.

```text
Creme brulee
```

**Starting code (already in the cell below):**

```python
import unicodedata

dish = "Crème brûlée"
```

<details><summary><b>Hint</b></summary>

Normalise to NFD so each accent becomes its own character. Accents have the Unicode category `"Mn"` (mark, non-spacing): `unicodedata.category(ch)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Creme brulee
```

</details>

<details><summary><b>Solution</b></summary>

```python
decomposed = unicodedata.normalize("NFD", dish)
plain = ""
for ch in decomposed:
    if unicodedata.category(ch) != "Mn":
        plain += ch
print(plain)
```

NFD splits `è` into `e` + a combining grave accent. The accents are all category `Mn`, so skipping them leaves the bare letters. Without the NFD step, `è` is a single character and nothing would be removed.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
decomposed = unicodedata.normalize("NFD", dish)
plain = "".join(ch for ch in decomposed if not unicodedata.combining(ch))
print(plain)
```

`unicodedata.combining(ch)` is non-zero for combining marks, and a generator with `join` avoids building the string one piece at a time. Use this for search keys only: in names, `è` and `e` are different letters, and stripping them changes the meaning.

</details>

In [ ]:
import unicodedata

dish = "Crème brûlée"

# your code here


### NB16-Q28 · L4 · Bitwise operators · Write

Compute a simple checksum for each message: XOR all of its ASCII bytes
together. Print each checksum as two hex digits, then whether changing one
digit changed the checksum.

```text
PAY 100: 59
PAY 900: 51
changed: True
```

**Starting code (already in the cell below):**

```python
original = "PAY 100"
tampered = "PAY 900"
```

<details><summary><b>Hint</b></summary>

Encode each message to bytes; looping over bytes gives ints. Start with `0` and use `^=` for each byte.

</details>

<details><summary><b>Expected output</b></summary>

```text
PAY 100: 59
PAY 900: 51
changed: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
def checksum(text):
    result = 0
    for byte in text.encode("ascii"):
        result ^= byte
    return result

first = checksum(original)
second = checksum(tampered)
print(f"{original}: {first:02x}")
print(f"{tampered}: {second:02x}")
print("changed:", first != second)
```

Looping over `bytes` gives ints, which XOR can combine. XOR flips the bits where the two values differ, so any single changed byte changes the result. `1` is `0x31` and `9` is `0x39`: they differ in one bit, and the checksum differs in that same bit.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import reduce
from operator import xor

def checksum(text):
    return reduce(xor, text.encode("ascii"), 0)

for message in (original, tampered):
    print(f"{message}: {checksum(message):02x}")
print("changed:", checksum(original) != checksum(tampered))
```

`reduce(xor, data, 0)` (NB10) is the folding loop in one call. XOR checksums only catch accidents: two changes can cancel out, and an attacker can fix them up. For tamper detection, seniors use `hashlib` or `hmac` (NB31).

</details>

In [ ]:
original = "PAY 100"
tampered = "PAY 900"

# your code here


### NB16-Q29 · L4 · unicode casefold · Fix

A street search should treat `STRASSE` and `Straße` as the same word, but
this prints `False`. Fix it.

**Buggy code (copied into the cell below):**

```python
typed = "STRASSE"
stored = "Straße"
print(typed.lower() == stored.lower())
```

<details><summary><b>Hint</b></summary>

`lower()` leaves `ß` alone. There's a stronger case-removing method from NB02.

</details>

<details><summary><b>Expected output</b></summary>

```text
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
typed = "STRASSE"
stored = "Straße"
print(typed.casefold() == stored.casefold())
```

`"Straße".lower()` is still `"straße"`, so it never matches `"strasse"`. `casefold()` is built for comparing text without case: it turns `ß` into `ss`, so both become `"strasse"`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import unicodedata

typed = "STRASSE"
stored = "Straße"
print(unicodedata.normalize("NFC", typed).casefold()
      == unicodedata.normalize("NFC", stored).casefold())
```

`lower()` is for display; `casefold()` is for comparing. Seniors combine it with NFC normalisation (Q25), which handles both case and the two-spellings problem in one key.

</details>

In [ ]:
typed = "STRASSE"
stored = "Straße"
print(typed.lower() == stored.lower())


### NB16-Q30 · L4 · Decimal · Write

Split `bill` between `people` so that the shares add up to exactly the bill.
Round each share **down** to the cent, and give any leftover cents to
person 1.

```text
Person 1: 33.34
Person 2: 33.33
Person 3: 33.33
Total: 100.00
```

**Starting code (already in the cell below):**

```python
from decimal import Decimal, ROUND_DOWN

bill = Decimal("100.00")
people = 3
```

<details><summary><b>Hint</b></summary>

`(bill / people).quantize(Decimal("0.01"), rounding=ROUND_DOWN)` is the base share. The leftover is `bill - share * people`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Person 1: 33.34
Person 2: 33.33
Person 3: 33.33
Total: 100.00
```

</details>

<details><summary><b>Solution</b></summary>

```python
share = (bill / people).quantize(Decimal("0.01"), rounding=ROUND_DOWN)
leftover = bill - share * people
shares = [share] * people
shares[0] = share + leftover
for number, amount in enumerate(shares, start=1):
    print(f"Person {number}: {amount}")
print("Total:", sum(shares))
```

`100 / 3` is `33.333...`; rounding *each* share to 33.33 loses a cent (3 × 33.33 = 99.99). Rounding down and handing the leftover cent to one person keeps the total exact. `sum` of `Decimal`s is exact too, so the check really prints `100.00`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
cents = int(bill * 100)
base, extra = divmod(cents, people)
shares = [Decimal(base + (1 if i < extra else 0)) / 100 for i in range(people)]
for number, amount in enumerate(shares, start=1):
    print(f"Person {number}: {amount}")
print("Total:", sum(shares))
```

Many payment systems work in integer cents: `divmod` gives the base share and how many people owe one extra cent, which also spreads the pennies fairly when the split is 7 ways. Never let rounding make money appear or disappear.

</details>

In [ ]:
from decimal import Decimal, ROUND_DOWN

bill = Decimal("100.00")
people = 3

# your code here


### NB16-Q31 · L4 · Bitwise operators · Refactor

This checks for powers of two with a loop. Refactor `is_power_of_two` to use
a bitwise trick instead, keeping the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def is_power_of_two(n):
    if n <= 0:
        return False
    while n % 2 == 0:
        n = n // 2
    return n == 1

for n in [1, 6, 64, 0, 96, 1024]:
    print(n, is_power_of_two(n))
```

<details><summary><b>Hint</b></summary>

A power of two has exactly one 1 bit (`1000`). Subtracting 1 flips it and every bit below it (`0111`). What does `n & (n - 1)` give then?

</details>

<details><summary><b>Expected output</b></summary>

```text
1 True
6 False
64 True
0 False
96 False
1024 True
```

</details>

<details><summary><b>Solution</b></summary>

```python
def is_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0

for n in [1, 6, 64, 0, 96, 1024]:
    print(n, is_power_of_two(n))
```

For a power of two, `n` and `n - 1` share no 1 bits, so `n & (n - 1)` is 0. Any other positive number has a second 1 bit that survives. The `n > 0` check handles 0 (and negatives), where the trick would give a wrong `True`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def is_power_of_two(n):
    return n > 0 and n.bit_count() == 1

for n in [1, 6, 64, 0, 96, 1024]:
    print(n, is_power_of_two(n))
```

`bit_count() == 1` says what it means: "exactly one bit set". The `n & (n - 1)` trick is classic and worth recognising in other people's code, but the readable version wins in new code.

</details>

In [ ]:
def is_power_of_two(n):
    if n <= 0:
        return False
    while n % 2 == 0:
        n = n // 2
    return n == 1

for n in [1, 6, 64, 0, 96, 1024]:
    print(n, is_power_of_two(n))


---

## L5 · Hard

### NB16-Q32 · L5 · Bitwise operators · Write

Convert `ip` to a single 32-bit integer using bit shifts, print it, then
convert it back to dotted text. Then, for each address in `checks`, print
whether it's inside the subnet `network/prefix` (the first `prefix` bits
must match).

```text
3232235853
192.168.1.77
192.168.1.77 in 192.168.1.0/24: True
192.168.2.5 in 192.168.1.0/24: False
```

**Starting code (already in the cell below):**

```python
ip = "192.168.1.77"
network = "192.168.1.0"
prefix = 24
checks = ["192.168.1.77", "192.168.2.5"]
```

<details><summary><b>Hint</b></summary>

Each of the 4 parts is one byte: shift the running value left by 8 and `|` in the next part. To go back, shift right by 24, 16, 8, 0 and mask with `0xFF`. The subnet mask is `prefix` ones followed by zeros, 32 bits wide.

</details>

<details><summary><b>Expected output</b></summary>

```text
3232235853
192.168.1.77
192.168.1.77 in 192.168.1.0/24: True
192.168.2.5 in 192.168.1.0/24: False
```

</details>

<details><summary><b>Solution</b></summary>

```python
def ip_to_int(text):
    value = 0
    for part in text.split("."):
        value = (value << 8) | int(part)
    return value

def int_to_ip(value):
    parts = [str((value >> shift) & 0xFF) for shift in (24, 16, 8, 0)]
    return ".".join(parts)

def in_subnet(address, network, prefix):
    mask = (0xFFFFFFFF << (32 - prefix)) & 0xFFFFFFFF
    return (ip_to_int(address) & mask) == (ip_to_int(network) & mask)

number = ip_to_int(ip)
print(number)
print(int_to_ip(number))
for address in checks:
    print(f"{address} in {network}/{prefix}: {in_subnet(address, network, prefix)}")
```

An IPv4 address is 4 bytes, so `<< 8` makes room for each new byte and `|` drops it in. The trap is the mask: in C, `0xFFFFFFFF << 8` loses the top bits, but Python ints never overflow, so the result grows to 40 bits. `& 0xFFFFFFFF` cuts it back to 32. Two addresses share a subnet when their masked values are equal.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import ipaddress

address = ipaddress.ip_address(ip)
print(int(address))
print(ipaddress.ip_address(int(address)))
subnet = ipaddress.ip_network(f"{network}/{prefix}")
for text in checks:
    print(f"{text} in {subnet}: {ipaddress.ip_address(text) in subnet}")
```

The `ipaddress` module does all of this, validates input (`999.1.1.1` raises `ValueError`), and handles IPv6. Seniors write the bit version once to understand it, then use the library.

</details>

In [ ]:
ip = "192.168.1.77"
network = "192.168.1.0"
prefix = 24
checks = ["192.168.1.77", "192.168.2.5"]

# your code here


### NB16-Q33 · L5 · struct · Write

`data` is a binary sales file. It starts with a 2-byte little-endian record
count, followed by that many records in the format `"<HHI"`: product id,
quantity, and unit price in **cents**. Print one line per record and the
total, with exact `Decimal` money:

```text
#101  3 x 19.99 =  59.97
#205 10 x  2.50 =  25.00
#310  1 x 149.00 = 149.00
Total: 233.97
```

The quantity is right-aligned in 2 characters, the price in 5 and the line
total in 6.

**Starting code (already in the cell below):**

```python
import struct
from decimal import Decimal

data = bytes.fromhex("0300" "65000300cf070000" "cd000a00fa000000" "36010100343a0000")
```

<details><summary><b>Hint</b></summary>

`struct.unpack_from(fmt, data, offset)` reads at a position. The header is `"<H"` (2 bytes) and each record is `struct.calcsize("<HHI")` bytes. Turn cents into money with `Decimal(cents) / 100`, then make sure it always shows 2 decimal places.

</details>

<details><summary><b>Expected output</b></summary>

```text
#101  3 x 19.99 =  59.97
#205 10 x  2.50 =  25.00
#310  1 x 149.00 = 149.00
Total: 233.97
```

</details>

<details><summary><b>Solution</b></summary>

```python
RECORD = "<HHI"
(count,) = struct.unpack_from("<H", data, 0)
size = struct.calcsize(RECORD)
total = Decimal(0)
for i in range(count):
    product, qty, cents = struct.unpack_from(RECORD, data, 2 + i * size)
    price = (Decimal(cents) / 100).quantize(Decimal("0.01"))
    line = price * qty
    total += line
    print(f"#{product} {qty:>2} x {price:>5} = {line:>6}")
print("Total:", total)
```

The header tells you how many records follow, and each record starts `size` (8) bytes after the previous one, so `2 + i * size` is its offset. `unpack` always returns a tuple, so `(count,) = ...` unpacks the single value. The trap is the price: `Decimal(250) / 100` is exactly right but prints as `2.5`, because `Decimal` drops zeros a division doesn't need. `quantize(Decimal("0.01"))` fixes it at 2 places, and then `2.50 * 10` keeps them: `25.00`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
HEADER = struct.Struct("<H")
RECORD = struct.Struct("<HHI")

(count,) = HEADER.unpack_from(data)
body = data[HEADER.size:HEADER.size + count * RECORD.size]
total = Decimal(0)
for product, qty, cents in RECORD.iter_unpack(body):
    price = Decimal(cents).scaleb(-2)
    line = price * qty
    total += line
    print(f"#{product} {qty:>2} x {price:>5} = {line:>6}")
print("Total:", total)
```

`iter_unpack` walks the records with no offset arithmetic, and slicing the body by the header's count means trailing junk is ignored. `scaleb(-2)` shifts the decimal point two places without division. Notice the price `149.00` overflows its 5-character column: a width is a minimum, not a limit.

</details>

In [ ]:
import struct
from decimal import Decimal

data = bytes.fromhex("0300" "65000300cf070000" "cd000a00fa000000" "36010100343a0000")

# your code here


### NB16-Q34 · L5 · UTF-8 · Write

An old system stores names in a field of `FIELD_BYTES` bytes of UTF-8.
For each name: normalise it to NFC, encode it, and if it's too long, cut
the bytes to fit **without** leaving half a character. Print the byte
length before cutting (right-aligned in 2 characters) and the stored name:

```text
 4 bytes -> Zoë
 6 bytes -> Renée
 8 bytes -> Dvoř
10 bytes -> Ελέ
```

`"Rene\u0301e"` arrives in decomposed (NFD) form: normalising first is
what lets it fit.

**Starting code (already in the cell below):**

```python
import unicodedata

FIELD_BYTES = 6
names = ["Zoë", "Rene\u0301e", "Dvořák", "Ελένη"]
```

<details><summary><b>Hint</b></summary>

`text.encode("utf-8")[:FIELD_BYTES]` may cut a 2-byte character in half. `.decode("utf-8", errors="ignore")` drops a broken piece at the end.

</details>

<details><summary><b>Expected output</b></summary>

```text
 4 bytes -> Zoë
 6 bytes -> Renée
 8 bytes -> Dvoř
10 bytes -> Ελέ
```

</details>

<details><summary><b>Solution</b></summary>

```python
for name in names:
    name = unicodedata.normalize("NFC", name)
    encoded = name.encode("utf-8")
    stored = encoded[:FIELD_BYTES].decode("utf-8", errors="ignore")
    print(f"{len(encoded):>2} bytes -> {stored}")
```

`Dvořák` is 8 bytes, and the first 6 end with the first half of `á`. Decoding that normally raises `UnicodeDecodeError`; `errors="ignore"` drops the broken half and leaves `Dvoř`. Greek letters take 2 bytes each, so 6 bytes hold exactly `Ελέ`. `Rene\u0301e` is 7 bytes in NFD form, so cutting it without normalising would lose the final `e`; in NFC it's exactly 6 bytes and fits whole.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def fit_utf8(text, limit):
    """NFC-normalise `text` and cut it to at most `limit` UTF-8 bytes."""
    encoded = unicodedata.normalize("NFC", text).encode("utf-8")
    return len(encoded), encoded[:limit].decode("utf-8", errors="ignore")

for name in names:
    size, stored = fit_utf8(name, FIELD_BYTES)
    print(f"{size:>2} bytes -> {stored}")
```

A named helper keeps the rule (normalise, encode, cut, decode) in one tested place. `errors="ignore"` is safe here only because the damage can only be at the very end; elsewhere, silently dropping bytes hides real corruption.

</details>

In [ ]:
import unicodedata

FIELD_BYTES = 6
names = ["Zoë", "Rene\u0301e", "Dvořák", "Ελένη"]

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB16-P · L5 · Mini-project: Invoice Engine · Write

Build an invoice from `items`. Each item is `(name, unit price text,
quantity, flags)`, and the flags are bits:

- `TAXABLE`: the line is taxed at `TAX_RATE`.
- `SALE`: 10% off. `CLEARANCE`: 25% off. If both are set, only
  `CLEARANCE` applies.

Work in `Decimal`. A line total is `price × quantity × (1 − discount)`,
rounded **half-up** to the cent. Tax is charged on the sum of the taxable
line totals, rounded half-up once. Print:

```text
* Notebook       3 x  4.99  -10%   13.47
* Fountain pen   1 x 24.50         24.50
  Bread          4 x  2.25          9.00
* Desk lamp      1 x 39.99  -25%   29.99
* Ink pack       2 x  6.75  -25%   10.13
Subtotal:   87.09
You saved:  14.87
Tax 8.25%:   6.44
Total:      93.53
(* taxable)
```

A `*` marks a taxable line. "You saved" is the full price of every line
minus the subtotal.

**Starting code (already in the cell below):**

```python
from decimal import Decimal, ROUND_HALF_UP

TAXABLE = 1
SALE = 2
CLEARANCE = 4
TAX_RATE = Decimal("0.0825")

items = [
    ("Notebook", "4.99", 3, TAXABLE | SALE),
    ("Fountain pen", "24.50", 1, TAXABLE),
    ("Bread", "2.25", 4, 0),
    ("Desk lamp", "39.99", 1, TAXABLE | CLEARANCE),
    ("Ink pack", "6.75", 2, TAXABLE | SALE | CLEARANCE),
]
```

<details><summary><b>Hint</b></summary>

Write a `discount(flags)` function that checks `CLEARANCE` before `SALE`. Keep a running subtotal, a running full price and a running taxable total. Round with `quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
* Notebook       3 x  4.99  -10%   13.47
* Fountain pen   1 x 24.50         24.50
  Bread          4 x  2.25          9.00
* Desk lamp      1 x 39.99  -25%   29.99
* Ink pack       2 x  6.75  -25%   10.13
Subtotal:   87.09
You saved:  14.87
Tax 8.25%:   6.44
Total:      93.53
(* taxable)
```

</details>

<details><summary><b>Solution</b></summary>

```python
CENT = Decimal("0.01")

def discount(flags):
    if flags & CLEARANCE:
        return Decimal("0.25")
    if flags & SALE:
        return Decimal("0.10")
    return Decimal("0")

subtotal = Decimal("0")
full_price = Decimal("0")
taxable = Decimal("0")
for name, price_text, qty, flags in items:
    price = Decimal(price_text)
    rate = discount(flags)
    line = (price * qty * (1 - rate)).quantize(CENT, rounding=ROUND_HALF_UP)
    subtotal += line
    full_price += price * qty
    mark = "*" if flags & TAXABLE else " "
    if flags & TAXABLE:
        taxable += line
    label = f"-{int(rate * 100)}%" if rate else ""
    print(f"{mark} {name:<14}{qty:>2} x {price:>5}  {label:>4}  {line:>6}")

tax = (taxable * TAX_RATE).quantize(CENT, rounding=ROUND_HALF_UP)
print(f"Subtotal:  {subtotal:>6}")
print(f"You saved: {full_price - subtotal:>6}")
print(f"Tax {TAX_RATE * 100:.2f}%: {tax:>6}")
print(f"Total:     {subtotal + tax:>6}")
print("(* taxable)")
```

Every amount starts as a `Decimal` built from text, so nothing is ever a float. Checking `CLEARANCE` first makes it win when both flags are set (the Ink pack). Each line is rounded half-up once, `6.75 × 2 × 0.75` = `10.125` becomes `10.13`, and the tax is rounded once on the taxable total, which is how real tills avoid a cent of drift per line.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
CENT = Decimal("0.01")
DISCOUNTS = [(CLEARANCE, Decimal("0.25")), (SALE, Decimal("0.10"))]

def to_cents(amount):
    return amount.quantize(CENT, rounding=ROUND_HALF_UP)

def discount(flags):
    """The first matching discount wins; CLEARANCE is listed first."""
    for flag, rate in DISCOUNTS:
        if flags & flag:
            return rate
    return Decimal(0)

subtotal = full_price = taxable = Decimal(0)
for name, price_text, qty, flags in items:
    price = Decimal(price_text)
    rate = discount(flags)
    line = to_cents(price * qty * (1 - rate))
    subtotal += line
    full_price += price * qty
    is_taxable = bool(flags & TAXABLE)
    taxable += line if is_taxable else 0
    label = f"-{rate:.0%}" if rate else ""
    print(f"{'*' if is_taxable else ' '} {name:<14}{qty:>2} x {price:>5}  "
          f"{label:>4}  {line:>6}")

tax = to_cents(taxable * TAX_RATE)
print(f"Subtotal:  {subtotal:>6}")
print(f"You saved: {full_price - subtotal:>6}")
print(f"Tax {TAX_RATE:.2%}: {tax:>6}")
print(f"Total:     {subtotal + tax:>6}")
print("(* taxable)")
```

The discount rules become data (`DISCOUNTS`, in priority order), so a new promotion is one line, not a new `if`. A `to_cents` helper names the single rounding rule. `Decimal` supports the `%` format code directly, so `f"{rate:.0%}"` prints `10%`. In NB21, `enum.Flag` would replace the bare integer flags with named, printable ones.

</details>

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

TAXABLE = 1
SALE = 2
CLEARANCE = 4
TAX_RATE = Decimal("0.0825")

items = [
    ("Notebook", "4.99", 3, TAXABLE | SALE),
    ("Fountain pen", "24.50", 1, TAXABLE),
    ("Bread", "2.25", 4, 0),
    ("Desk lamp", "39.99", 1, TAXABLE | CLEARANCE),
    ("Ink pack", "6.75", 2, TAXABLE | SALE | CLEARANCE),
]

# your code here


---

## Done

Next up: **NB17 · Regex** in `intermediate/`.